# Ensnarlment in physical networks
This is to explore the ensnarlment between networks in synthetic lattices

Prerequisite:
- Package "Plotly" for 3D plots
- Package "kaleido" for saving 3D plots

## Imports

All analysis & visualization routines now live in the **`gemini_ensnarl`** package (see `gemini_ensnarl/`). Install it from the repository.

In [ ]:
import numpy as np
import networkx as nx
import pandas as pd
from itertools import combinations

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib import colormaps

import plotly
import plotly.graph_objects as go
import plotly.io as pio

from gemini_ensnarl import (
    Gauss_linking_integral,
    build_sibigraph,
    critical_flip_angles,
    candidate_mid_angles,
    sign_patterns_from_angles,
    generate_periodic_lattices,
    generate_ladder_lattices,
)
from gemini_ensnarl.viz_lattice import *


## Main - step lattices

In [ ]:
### Create spatial networks
num_periods = 2
G1, G2 = generate_periodic_lattices(num_periods)

# plot with labels
# plot_networkx_dual(D)
# Graph 1
pos1 = nx.get_node_attributes(G1, 'pos')
# Graph 2
pos2 = nx.get_node_attributes(G2, 'pos')
# camera
camera = dict(
    eye=dict(x=-0.6, y=1.9, z=.7),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)

lab_G = 'step-p{}'.format(num_periods)

plot_networkx_12label(G1, G2, pos1, pos2, camera=camera, lab_name=lab_G+'-G',figsize=(700, 700))

In [ ]:
# plot one without label
plot_networkx_12label(G1, G2, pos1, pos2, camera=camera, lab_name=lab_G+'-G-nolab', nlabel=False)

In [ ]:
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # record the position
        e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
        # direct computation of Gauss linking integral
        gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
        Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
val_max = np.abs(Lam_gli).max()
print("max abs value in Lam_gli:", val_max)

plt.figure(figsize=(6, 3.5))
plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
# Set tick positions (match number of labels)
xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli.png".format(lab_G), dpi=300, bbox_inches='tight')
plt.show()

### Signed nature

In [ ]:
labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
labels_p2 = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]

Gb = build_sibigraph(Lam_gli, labels_p1, labels_p2)

In [ ]:
figsize = (9.5,4) # (4.5, 4) for p=1
draw_sibigraph(Gb, figname=lab_G, figsize=figsize, lab_diff=0.07)

In [ ]:
# check the balance
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

deg = np.sum(abs(W), axis=1)
# D_inv = np.diag(1./deg)
# P = D_inv.dot(W)
# eigvals = np.linalg.eigvals(P)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# check the eigenvectors
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

deg = np.sum(abs(W), axis=1)
D = np.diag(deg)

L = D - W
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals))

# indices of the smallest value
tol = 1e-4
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L:", eigvals[idx])

# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
# xticks = edges1.copy()
# xticks.extend(edges2)
xticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
xticks.extend(['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2])
plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
plt.grid(True)
plt.show()

In [ ]:
# check the eigenvectors - normalized Laplacian
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

deg = np.sum(abs(W), axis=1)
D2_inv = np.diag(1./np.sqrt(deg))
L_sym = np.eye(W.shape[0]) - D2_inv.dot(W).dot(D2_inv)

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-4
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])

# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')
# xticks = edges1.copy()
# xticks.extend(edges2)
xticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
xticks.extend(['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2])
plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)

plt.grid(True)
plt.show()

This provides a degenerate case when there are more than one eigenvector corresponding to the smallest eigenvalue of the Laplacian.

In [ ]:
# choose the eigenvector of L
from itertools import combinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol = 1e-4
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("Flip angles:", flip_angles)
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = None
best_partition = None

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost:
            best_cost = cost
            best_theta = theta
            best_partition = (s)
        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif cost == best_cost:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif cost == best_cost:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif cost == best_cost:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost)
print("Best partition:", best_partition)

In [ ]:
# We have two equivalent clustering results and let's have a look!
lab_p = ''

i_c = 0
for s in best_partition:
    nodec = [1 if s[i] > 0 else 0 for i in range(len(s))]
    # Partition
    # draw_sibigraph_nodec(Gb, nodec, cmap_name='coolwarm', figname=lab_G+lab_p+'-s{}'.format(i_c), figsize=figsize)
    # Switching
    # update the labels
    node_c = [nodec][0]

    # s = [1 if node_c[i] == 1 else -1 for i in range(len(node_c))]
    S = np.diag(s)
    W_s = S.dot(W.dot(S))
    B_s = W_s[:len(edges1), :][:, -len(edges2):]

    labels_p1s = []
    labels_p2s = []
    # reverse the edge if node_c[i]=-1
    for i in range(len(edges1)):
        if node_c[i] == 1:
            labels_p1s.append(labels_p1[i])
        else:
            e = edges1[i]
            labels_p1s.append('({},{})'.format(e[1], e[0]))
    for j in range(len(edges2)):
        if node_c[j+len(edges1)] == 1:
            labels_p2s.append(labels_p2[j])
        else:
            e = edges2[j]
            labels_p2s.append('({},{})'.format(e[1]+G1.number_of_nodes(), e[0]+G1.number_of_nodes()))

    Gb_s = build_sibigraph(B_s, labels_p1s, labels_p2s)
    draw_sibigraph(Gb_s, figname=lab_G+'-switch'+lab_p+'-s{}'.format(i_c), figsize=figsize)
    draw_sibigraph_node2c(Gb_s, node_c, figname=lab_G+'-switch'+lab_p+'-s{}'.format(i_c), figsize=figsize)

    i_c += 1

#### Check-ups

In [ ]:
# check the exact solution -- only if small!!!
tol = 1e-4

gbest_cost = np.inf
gbest_partition = None

size = L.shape[0]
s = np.ones(size)
best_partition = []
for num in range(1, int((size+1.)/2. + 1)):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[list(idx_c)] = -1
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s_now)

print("Best cost:", best_cost)
print("Best partition:", best_partition)

In [ ]:
# s = best_partition[0]

idx_c = 0
for s in best_partition:

    nodec = [1 if s[i] > 0 else 0 for i in range(len(s))]
    # Partition
    # draw_sibigraph_nodec(Gb, nodec, cmap_name='coolwarm', figname=lab_G+lab_p+'-exact{}'.format(idx_c), figsize=figsize)
    # Switching
    # update the labels
    node_c = [nodec][0]

    # s = [1 if node_c[i] == 1 else -1 for i in range(len(node_c))]
    S = np.diag(s)
    W_s = S.dot(W.dot(S))
    B_s = W_s[:len(edges1), :][:, -len(edges2):]

    labels_p1s = []
    labels_p2s = []
    for i in range(len(edges1)):
        if node_c[i] == 1:
            labels_p1s.append(labels_p1[i])
        else:
            e = edges1[i]
            labels_p1s.append('({},{})'.format(e[1], e[0]))
    for j in range(len(edges2)):
        if node_c[j+len(edges1)] == 1:
            labels_p2s.append(labels_p2[j])
        else:
            e = edges2[j]
            labels_p2s.append('({},{})'.format(e[1]+G1.number_of_nodes(), e[0]+G1.number_of_nodes()))

    Gb_s = build_sibigraph(B_s, labels_p1s, labels_p2s)
    draw_sibigraph(Gb_s, figname=lab_G+'-switch'+lab_p+'-exact{}'.format(idx_c), figsize=figsize)
    draw_sibigraph_node2c(Gb_s, node_c, figname=lab_G+'-switch'+lab_p+'-exact{}'.format(idx_c), figsize=figsize)

    idx_c += 1

Actually, the two solutions are just two eigenvectors that are returned by the symmetric Laplacian!

### Linking centrality

In [ ]:
# compute the linking centrality
cmap = 'plasma'
matrix = Lam_gli.copy()
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(matrix).sum(axis=1)
# For G2: absolute column sum (one per edge)
col_sums = np.abs(matrix).sum(axis=0)

pos1 = nx.get_node_attributes(G1, 'pos')
# Graph 2
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sums, col_sums, camera=camera, lab_name=lab_G+'-cent', cmap=cmap, xval=.8)

### Disturbing the cyclic structure

#### One and the other
To be added: just removing edges from one lattice

In [ ]:
### Create spatial networks
num_periods = 1
G1, G2 = generate_periodic_lattices(num_periods)

# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # record the position
        e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
        # direct computation of Gauss linking integral
        gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
        Lam_gli[i, j] = gli_ij


# centrality
matrix = Lam_gli.copy()
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(matrix).sum(axis=1)
# For G2: absolute column sum (one per edge)
col_sums = np.abs(matrix).sum(axis=0)

In [ ]:
# removing edges according to their centrality - G1 & G2
row_sort = np.argsort(row_sums)[::-1]
col_sort = np.argsort(col_sums)[::-1]

res = []

for idr in range(len(row_sort)):
    res.append([])
    # delete the corresponding rows
    Lam_row = Lam_gli.copy()
    Lam_row = np.delete(Lam_row, row_sort[:idr], axis=0)

    for idc in range(len(col_sort)):
        print("remove {} rows, {} columns".format(idr, idc))

        Lam_col = Lam_row.copy()
        Lam_col = np.delete(Lam_col, col_sort[:idc], axis=1)


        # the unbalance score: smallest eigenvalue of normalized Laplacian
        W = np.block([[np.zeros((Lam_col.shape[0], Lam_col.shape[0])), Lam_col], [Lam_col.T, np.zeros((Lam_col.shape[1], Lam_col.shape[1]))]])
        deg = np.sum(abs(W), axis=1)
        deg_inv = deg.copy()
        deg_inv[deg_inv > 0] = 1./deg_inv[deg_inv > 0]
        D2_inv = np.diag(np.sqrt(deg_inv))
        I_diag = np.zeros(W.shape[0])
        I_diag[deg_inv > 0] = 1.
        L_sym = np.diag(I_diag) - D2_inv.dot(W).dot(D2_inv)

        eigvals = np.linalg.eigvalsh(L_sym)
        eigvals_sort = np.sort(eigvals)
        # compute the number of connected components
        Gbi_now = nx.from_numpy_array(W)
        num_components = nx.number_connected_components(Gbi_now)
        print("Number of connected components:", num_components)
        print("Min-{} eigenvalues of L sym:".format(num_components), eigvals_sort[:num_components])

        # we sum over the least num_components eigenvalues
        res[-1].append(sum(eigvals_sort[:num_components]))

In [ ]:
print("min, max values", np.min(res), np.max(res))
vmin = 0.
vmax = 0.3907
tick_int = 1

plt.figure(figsize=(5.2, 3.3))
plt.imshow(res, cmap='Reds', vmin=vmin, vmax=vmax)

xticks = np.arange(G2.number_of_edges())
plt.xticks(ticks=xticks[::tick_int], labels=xticks[::tick_int])
yticks = np.arange(G1.number_of_edges())
plt.yticks(ticks=yticks[::tick_int], labels=yticks[::tick_int])

plt.colorbar(shrink=0.8) #shrink=0.6
plt.savefig("step-p{}-remv-edges-G12-uni.png".format(num_periods), dpi=300, bbox_inches='tight')
plt.show()

#### Together
What if we mix the two graphs and always remove the edge of the highest linking centrality, no matter where graph it belongs?

In [ ]:
### Create spatial networks
num_periods = 2
G1, G2 = generate_periodic_lattices(num_periods)

# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # record the position
        e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
        # direct computation of Gauss linking integral
        gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
        Lam_gli[i, j] = gli_ij

In [ ]:
# initial centrality
matrix = Lam_gli.copy()
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(matrix).sum(axis=1)

# For G2: absolute column sum (one per edge)
col_sums = np.abs(matrix).sum(axis=0)

# visualization
plt.figure(figsize=(6, 4))
plt.plot(row_sums, c='r', marker='o', linestyle='dashed')
plt.plot(col_sums, c='b', marker='*', linestyle='dashed')
plt.grid()
plt.legend(['G1', 'G2'])
# plt.savefig("step-p{}-centrality.png".format(num_periods), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### initial value ###
# the unbalance score: smallest eigenvalue of normalized Laplacian
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])
deg = np.sum(abs(W), axis=1)
D2_inv = np.diag(1./np.sqrt(deg))
L_sym = np.eye(W.shape[0]) - D2_inv.dot(W).dot(D2_inv)
eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# # initial centrality
matrix = Lam_gli.copy()
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(matrix).sum(axis=1)
# For G2: absolute column sum (one per edge)
col_sums = np.abs(matrix).sum(axis=0)
idr_max = np.argmax(row_sums)
idc_max = np.argmax(col_sums)
print("row max:", idr_max, "max_val:", row_sums[idr_max])
print("column max:", idc_max, "max_val:", col_sums[idc_max])
# rc_nz = np.array([sum(row_sums > 0), sum(col_sums > 0)])
cent = np.concatenate([row_sums, col_sums], axis=None)

# removing edges from the GLI
G1_now = G1.copy()
G2_now = G2.copy()
edges_remv = ['']
edges_cent = [np.nan]
Lam_now = Lam_gli.copy()
bal_vals = [min(eigvals)]
cen_meas = [np.mean(cent)]
cen_stds = [np.std(cent)]
cen_max = [np.max(cent)]
cyc_cout = [(len(nx.cycle_basis(G1_now)), len(nx.cycle_basis(G2_now)))]

idx_row = list(range(len(edges1)))
idx_col = list(range(len(edges2)))

count = 0
while ((len(idx_row) > 1) and (len(idx_col) > 1)): # when there is at least one nonzero row/column

    # "delete" the corresponding row/column
    if row_sums[idr_max] >= col_sums[idc_max]:
        print("remove row:", idr_max)
        idx_row.remove(idr_max)
        # set the corresponding row to zero
        Lam_now[idr_max, :] = 0
        # record the removed edge
        edges_remv.append(edges1[idr_max])
        edges_cent.append(row_sums[idr_max])
        print("edge remove:", edges_remv[-1], "centrality:", edges_cent[-1])
        # the number of cycles in G1
        G1_now.remove_edge(edges_remv[-1][0], edges_remv[-1][1])
    else:
        print("remove column:", idc_max)
        idx_col.remove(idc_max)
        # set the corresponding column to zero
        Lam_now[:, idc_max] = 0
        # record the removed edge
        edges_remv.append(edges2[idc_max])
        edges_cent.append(col_sums[idc_max])
        print("edge remove:", edges_remv[-1], "centrality:", edges_cent[-1])
        # the number of cycles in G1
        G2_now.remove_edge(edges_remv[-1][0], edges_remv[-1][1])

    count += 1
    # !! plot the two networks !!
    # Graph 1
    pos1 = nx.get_node_attributes(G1_now, 'pos')
    # Graph 2
    pos2 = nx.get_node_attributes(G2_now, 'pos')
    plot_networkx_12label(G1_now, G2_now, pos1, pos2, camera=camera, lab_name=lab_G+'-G-rem{}'.format(count))

    cyc_cout.append((len(nx.cycle_basis(G1_now)), len(nx.cycle_basis(G2_now))))

    # the unbalance score: smallest eigenvalue of normalized Laplacian
    W = np.block([[np.zeros((Lam_now.shape[0], Lam_now.shape[0])), Lam_now], [Lam_now.T, np.zeros((Lam_now.shape[1], Lam_now.shape[1]))]])
    deg = np.sum(abs(W), axis=1)
    deg_inv = deg.copy()
    deg_inv[deg_inv > 0] = 1./deg_inv[deg_inv > 0]
    D2_inv = np.diag(np.sqrt(deg_inv))
    I_diag = np.zeros(W.shape[0])
    I_diag[deg_inv > 0] = 1.
    L_sym = np.diag(I_diag) - D2_inv.dot(W).dot(D2_inv)

    eigvals = np.linalg.eigvalsh(L_sym)
    eigvals_sort = np.sort(eigvals)
    # compute the number of connected components
    Gbi_now = nx.from_numpy_array(W)
    num_components = nx.number_connected_components(Gbi_now)
    print("Number of connected components:", num_components)
    print("Min-{} eigenvalues of L sym:".format(num_components), eigvals_sort[:num_components])
    # we sum over the least num_components eigenvalues
    bal_vals.append(sum(eigvals_sort[:num_components]))

    # update the centrality
    matrix = Lam_now.copy()
    row_sums = np.abs(matrix).sum(axis=1)
    col_sums = np.abs(matrix).sum(axis=0)
    cent = np.concatenate([row_sums[idx_row], col_sums[idx_col]], axis=None)

    # visualize the graph with centrality G1_now, G2_now, pos1, pos2, camera=camera, lab_name=lab_G+'-G-rem{}'.format(count)
    plot_networkx_colore(G1_now, G2_now, pos1, pos2, row_sums[idx_row], col_sums[idx_col],
                         camera=camera, lab_name=lab_G+'-cent-rem{}'.format(count), cmap=cmap, xval=.8)

    cen_meas.append(np.mean(cent))
    cen_stds.append(np.std(cent))
    cen_max.append(np.max(cent))

    # update the max index
    idr_max = np.argmax(row_sums)
    idc_max = np.argmax(col_sums)

    # stop if all zero
    if (row_sums[idr_max] == 0) | (col_sums[idc_max] == 0):
        print("Only zero values!")
        break

##### For Figures

In [ ]:
# plot the first four with the same color scale
# set the same scale
cmin = 0.0674
cmax = 0.7815
# cmin = 0.05
# cmax = 0.80
nplot = 4

### initial value ###
# # initial centrality
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(Lam_gli).sum(axis=1)
# For G2: absolute column sum (one per edge)
col_sums = np.abs(Lam_gli).sum(axis=0)
idr_max = np.argmax(row_sums)
idc_max = np.argmax(col_sums)
print("row max:", idr_max, "max_val:", row_sums[idr_max])
print("column max:", idc_max, "max_val:", col_sums[idc_max])
# rc_nz = np.array([sum(row_sums > 0), sum(col_sums > 0)])
cent = np.concatenate([row_sums, col_sums], axis=None)

# removing edges from the GLI
G1_now = G1.copy()
G2_now = G2.copy()
edges_remv = ['']
edges_cent = [np.nan]
Lam_now = Lam_gli.copy()
row_nows = row_sums.copy()
col_nows = col_sums.copy()
cen_meas = [np.mean(cent)]
cen_stds = [np.std(cent)]
cen_max = [np.max(cent)]
cyc_cout = [(len(nx.cycle_basis(G1_now)), len(nx.cycle_basis(G2_now)))]

idx_row = list(range(len(edges1)))
idx_col = list(range(len(edges2)))

count = 0
while count < nplot: # when there is at least one nonzero row/column

    # "delete" the corresponding row/column
    if row_nows[idr_max] >= col_nows[idc_max]:
        print("remove row:", idr_max)
        idx_row.remove(idr_max)
        # set the corresponding row to zero
        Lam_now[idr_max, :] = 0
        # record the removed edge
        edges_remv.append(edges1[idr_max])
        edges_cent.append(row_nows[idr_max])
        print("edge remove:", edges_remv[-1], "centrality:", edges_cent[-1])
        # the number of cycles in G1
        G1_now.remove_edge(edges_remv[-1][0], edges_remv[-1][1])
    else:
        print("remove column:", idc_max)
        idx_col.remove(idc_max)
        # set the corresponding column to zero
        Lam_now[:, idc_max] = 0
        # record the removed edge
        edges_remv.append(edges2[idc_max])
        edges_cent.append(col_nows[idc_max])
        print("edge remove:", edges_remv[-1], "centrality:", edges_cent[-1])
        # the number of cycles in G1
        G2_now.remove_edge(edges_remv[-1][0], edges_remv[-1][1])

    count += 1
    # !! plot the two networks !!
    # Graph 1
    pos1 = nx.get_node_attributes(G1_now, 'pos')
    # Graph 2
    pos2 = nx.get_node_attributes(G2_now, 'pos')
    # plot_networkx_12label(G1_now, G2_now, pos1, pos2, camera=camera, lab_name=lab_G+'-G-rem{}'.format(count))

    cyc_cout.append((len(nx.cycle_basis(G1_now)), len(nx.cycle_basis(G2_now))))


    # update the centrality
    row_nows = np.abs(Lam_now).sum(axis=1)
    col_nows = np.abs(Lam_now).sum(axis=0)
    print("row cents:", row_nows)
    print("col cents:", col_nows)
    cent = np.concatenate([row_nows[idx_row], col_nows[idx_col]], axis=None)

    # visualize the graph with centrality G1_now, G2_now, pos1, pos2, camera=camera, lab_name=lab_G+'-G-rem{}'.format(count)
    plot_networkx_colore(G1_now, G2_now, pos1, pos2, row_nows[idx_row], col_nows[idx_col],
                         camera=camera, lab_name=lab_G+'-cent-rem{}-uni'.format(count),
                         val_range=[cmin, cmax], cmap=cmap, xval=.8)

    cen_meas.append(np.mean(cent))
    cen_stds.append(np.std(cent))
    cen_max.append(np.max(cent))

    # update the max index
    idr_max = np.argmax(row_nows)
    idc_max = np.argmax(col_nows)

    # stop if all zero
    if (row_nows[idr_max] == 0) | (col_nows[idc_max] == 0):
        print("Only zero values!")
        break

In [ ]:
# compute the linking centrality
matrix = Lam_gli.copy()
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(matrix).sum(axis=1)
# For G2: absolute column sum (one per edge)
col_sums = np.abs(matrix).sum(axis=0)

plot_networkx_colore(G1, G2, pos1, pos2, row_sums, col_sums, camera=camera, val_range=[cmin, cmax], lab_name=lab_G+'-cent-uni', cmap=cmap, xval=.8)

## Main - ladder lattices

In [ ]:
### Create spatial networks (interlaced ladder / "catenation" lattices)
num_periods = 2
G1, G2 = generate_ladder_lattices(num_periods)

graph_sets = [G1, G2]
G1 = graph_sets[0].copy()
G2 = graph_sets[1].copy()

# Graph 1
pos1 = nx.get_node_attributes(G1, 'pos')
# Graph 2
pos2 = nx.get_node_attributes(G2, 'pos')

lab_G = 'lad-p{}'.format(num_periods)


In [ ]:
figsize_g=(800, 420) # (600, 420) for p=1; (800, 420) for p=2;

camera = dict(
    eye=dict(x=0.6, y=1.76, z=.39),
    center=dict(x=0, y=0, z=0),
    up=dict(x=0, y=0, z=.1)
)

# plot_networkx_12label(G1, G2, pos1, pos2)
plot_networkx_12label(G1, G2, pos1, pos2, camera=camera, lab_name=lab_G+'-G', figsize=figsize_g)

In [ ]:
# plot one without label
plot_networkx_12label(G1, G2, pos1, pos2, camera=camera, lab_name=lab_G+'-G-nolab', nlabel=False, figsize=figsize_g)

In [ ]:
# compute the GLI operator
# GLI between edges directly
edges1 = list(G1.edges())
edges2 = list(G2.edges())

Lam_gli = np.zeros((len(edges1), len(edges2)))
for i in range(Lam_gli.shape[0]):
    e1 = edges1[i]
    e1_pos = [np.array(G1.nodes[e1[0]]['pos']), np.array(G1.nodes[e1[1]]['pos'])]
    for j in range(Lam_gli.shape[1]):
        e2 = edges2[j]
        # record the position
        e2_pos = [np.array(G2.nodes[e2[0]]['pos']), np.array(G2.nodes[e2[1]]['pos'])]
        # direct computation of Gauss linking integral
        gli_ij = Gauss_linking_integral(e1_pos, e2_pos)
        Lam_gli[i, j] = gli_ij

In [ ]:
# Visualization
val_max = np.abs(Lam_gli).max()
print("max abs value in Lam_gli:", val_max)

plt.figure(figsize=(5, 3.))
plt.imshow(Lam_gli, cmap='seismic', vmax=val_max, vmin=-val_max) #, aspect='auto'
# Set tick positions (match number of labels)
xticks = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]
yticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
plt.yticks(ticks=np.arange(len(yticks)), labels=yticks)

# Optional: show grid or colorbar
plt.grid(False)
plt.colorbar(shrink=0.6)
plt.savefig("{}-Lam_gli.png".format(lab_G), dpi=300, bbox_inches='tight')
plt.show()

### Signed nature

In [ ]:
labels_p1 = ['({},{})'.format(e[0], e[1]) for e in edges1]
labels_p2 = ['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2]

Gb = build_sibigraph(Lam_gli, labels_p1, labels_p2)

In [ ]:
figsize_b = (6, 4) # (8, 4) for p=2; (6, 4) for p=1 (ladder)
# (9.5,4) for p=2 (4.5, 4) for p=1 (periodic)
draw_sibigraph(Gb, figname=lab_G, figsize=figsize_b, lab_diff=0.07)

In [ ]:
# check the balance
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

deg = np.sum(abs(W), axis=1)
# D_inv = np.diag(1./deg)
# P = D_inv.dot(W)
# eigvals = np.linalg.eigvals(P)

# symmetric version
D2_inv = np.diag(1./np.sqrt(deg))
P_sym = D2_inv.dot(W).dot(D2_inv)
eigvals = np.linalg.eigvals(P_sym)
print("Max eigenval of P:", np.max(eigvals))
# print("eigenvalues of P:", eigvals)

In [ ]:
# check the eigenvectors
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

deg = np.sum(abs(W), axis=1)
D = np.diag(deg)

L = D - W
eigvals, eigvecs = np.linalg.eigh(L)
print("Min eigenvalues of L:", min(eigvals))

# indices of the smallest value
tol = 1e-4
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

# choose the eigenvector(s)
plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')

# xticks = edges1.copy()
# xticks.extend(edges2)
xticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
xticks.extend(['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2])
plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

In [ ]:
# check the eigenvectors - normalized Laplacian
W = np.block([[np.zeros((Lam_gli.shape[0], Lam_gli.shape[0])), Lam_gli], [Lam_gli.T, np.zeros((Lam_gli.shape[1], Lam_gli.shape[1]))]])

deg = np.sum(abs(W), axis=1)
D2_inv = np.diag(1./np.sqrt(deg))
L_sym = np.eye(W.shape[0]) - D2_inv.dot(W).dot(D2_inv)

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L sym:", min(eigvals))

# indices of the smallest value
tol = 1e-4
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol]
print("Min eigenvalue(s) of L sym:", eigvals[idx])
print("first few eigenvalues:", eigvals[:5])

# choose the eigenvector(s)
plt.figure(figsize=(18, 5))
plt.subplot(121)
# choose the eigenvector(s)
for i in idx:
    plt.plot(eigvecs[:, i], marker='.', linestyle='dashed')

# xticks = edges1.copy()
# xticks.extend(edges2)
xticks = ['({},{})'.format(e[0], e[1]) for e in edges1]
xticks.extend(['({},{})'.format(e[0]+G1.number_of_nodes(), e[1]+G1.number_of_nodes()) for e in edges2])
plt.xticks(ticks=np.arange(len(xticks)), labels=xticks, rotation=60)
plt.grid(True)

plt.subplot(122)
plt.plot(eigvals, marker='.', linestyle='dashed')
plt.grid(True)
plt.show()

Bi-clustering 1: choose the smallest eigenvalue:

There are zeros in the code, we should separate into different cases (p=1)

In [ ]:
### Bi-Clustering ###
tol = 1e-4
nodec = []
idx_0s = []

for i,val in enumerate(eigvecs[:, idx[0]]):
    if val > tol:
        nodec.append(2)
    elif val < -tol:
        nodec.append(0)
    else:
        idx_0s.append(i)
        nodec.append(1)

if len(idx_0s) == 0:
    print("clear cut!")
    nodec = [1 if nodec[i] > 0 else 0 for i in range(len(nodec))]
else:
    print("# zeros in the eigenvector:", len(idx_0s))

In [ ]:
# allocation zeros
from itertools import combinations

s = np.array([1. if nodec[i] > 0 else -1. for i in range(len(nodec))]) # Default: all zeros in the same group as negatives
best_cost = s.T.dot(L).dot(s)
best_partition = [s]
size = len(idx_0s)
idx_0s = np.array(idx_0s)

res = []
for num in range(1, int((size+1.)/2.)+1):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[idx_0s[list(idx_c)]] = -1 # allocate these edges to the -1-group
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            print("change to -1:", idx_c)
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost)
print("# Best partition:", len(best_partition))

In [ ]:
# We have two equivalent clustering results and let's have a look!
lab_p = ''

i_c = 0
for s in best_partition:
    nodec = [1 if s[i] > 0 else 0 for i in range(len(s))]
    # Partition
    # draw_sibigraph_nodec(Gb, nodec, cmap_name='coolwarm', figname=lab_G+lab_p+'-s{}'.format(i_c), figsize=figsize)
    # Switching
    # update the labels
    node_c = [nodec][0]

    # s = [1 if node_c[i] == 1 else -1 for i in range(len(node_c))]
    S = np.diag(s)
    W_s = S.dot(W.dot(S))
    B_s = W_s[:len(edges1), :][:, -len(edges2):]

    labels_p1s = []
    labels_p2s = []
    # reverse the edge if node_c[i]=-1
    for i in range(len(edges1)):
        if node_c[i] == 1:
            labels_p1s.append(labels_p1[i])
        else:
            e = edges1[i]
            labels_p1s.append('({},{})'.format(e[1], e[0]))
    for j in range(len(edges2)):
        if node_c[j+len(edges1)] == 1:
            labels_p2s.append(labels_p2[j])
        else:
            e = edges2[j]
            labels_p2s.append('({},{})'.format(e[1]+G1.number_of_nodes(), e[0]+G1.number_of_nodes()))

    Gb_s = build_sibigraph(B_s, labels_p1s, labels_p2s)
    draw_sibigraph(Gb_s, figname=lab_G+'-switch'+lab_p+'-s{}'.format(i_c), figsize=figsize_b)
    draw_sibigraph_node2c(Gb_s, node_c, figname=lab_G+'-switch'+lab_p+'-s{}'.format(i_c), figsize=figsize_b)

    i_c += 1

Bi-clustering 2: combine smallest eigenvectors

In [ ]:
# Combine the first two eigenvectors: they are much closer to each other compared with others
tol = 1e-4

from itertools import combinationscombinations

eigvals, eigvecs = np.linalg.eigh(L_sym)
print("Min eigenvalues of L_sym:", min(eigvals)) # !!! L_sym

# indices of the smallest value
tol_ev = 0.05
eval_min = min(eigvals)
idx = [i for i, val in enumerate(eigvals) if abs(val-eval_min) < tol_ev]
print("Min eigenvalue(s) of L_sym:", eigvals[idx])

u1 = eigvecs[:, idx[0]]
u2 = eigvecs[:, idx[1]]

flip_angles, nodes_deg = critical_flip_angles(u1, u2)
print("#Flip angles:", len(flip_angles))
theta_candidates = candidate_mid_angles(flip_angles)

# Get sign patterns for all candidates
S = sign_patterns_from_angles(u1, u2, theta_candidates)  # shape (n, m)

# For each candidate, evaluate your signed cut cost C(theta_j)
best_cost = np.inf
best_theta = []
best_partition = []

for j, theta in enumerate(theta_candidates):
    s = S[:, j]         # +-1 labels
    # 0 entries?
    idx_0 = np.where(s == 0)[0]
    if len(idx_0) > 0:
        print("0 occurs in eigenvalues")
        ## try all combination of assignments
        # all in the negative one
        s[idx_0] = -1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

        # all other combinations
        for num in range(1, len(idx_0)):
            for idx_01 in combinations(idx_0, num):
                s_now = s.copy()
                s_now[idx_0] = -1
                s_now[list(idx_01)] = 1
                cost = s_now.T.dot(L).dot(s_now)
                if cost < best_cost-tol:
                    best_cost = cost
                    best_theta = [theta]
                    best_partition = [s_now]
                elif abs(cost-best_cost) < tol:
                    best_theta.append(theta)
                    best_partition.append(s_now)
        # the last one
        s[idx_0] = 1
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)
    else:
        # compute the objective
        cost = s.T.dot(L).dot(s)
        if cost < best_cost-tol:
            best_cost = cost
            best_theta = [theta]
            best_partition = [s]
        elif abs(cost-best_cost) < tol:
            best_theta.append(theta)
            best_partition.append(s)

print("Best theta:", best_theta)
print("Best cost:", best_cost)
print("# Best partitions:", len(best_partition))

In [ ]:
# We have two equivalent clustering results and let's have a look!
lab_p = 'v2'

i_c = 0
for s in best_partition:
    nodec = [1 if s[i] > 0 else 0 for i in range(len(s))]
    # Partition
    # draw_sibigraph_nodec(Gb, nodec, cmap_name='coolwarm', figname=lab_G+lab_p+'-s{}'.format(i_c), figsize=figsize)
    # Switching
    # update the labels
    node_c = [nodec][0]

    # s = [1 if node_c[i] == 1 else -1 for i in range(len(node_c))]
    S = np.diag(s)
    W_s = S.dot(W.dot(S))
    B_s = W_s[:len(edges1), :][:, -len(edges2):]

    labels_p1s = []
    labels_p2s = []
    # reverse the edge if node_c[i]=-1
    for i in range(len(edges1)):
        if node_c[i] == 1:
            labels_p1s.append(labels_p1[i])
        else:
            e = edges1[i]
            labels_p1s.append('({},{})'.format(e[1], e[0]))
    for j in range(len(edges2)):
        if node_c[j+len(edges1)] == 1:
            labels_p2s.append(labels_p2[j])
        else:
            e = edges2[j]
            labels_p2s.append('({},{})'.format(e[1]+G1.number_of_nodes(), e[0]+G1.number_of_nodes()))

    Gb_s = build_sibigraph(B_s, labels_p1s, labels_p2s)
    draw_sibigraph(Gb_s, figname=lab_G+'-switch'+lab_p+'-s{}'.format(i_c), figsize=figsize_b)
    draw_sibigraph_node2c(Gb_s, node_c, figname=lab_G+'-switch'+lab_p+'-s{}'.format(i_c), figsize=figsize_b)

    i_c += 1

#### Check-ups

In [ ]:
# check the exact solution -- only if small!!!
tol = 1e-4

gbest_cost = np.inf
gbest_partition = None

size = L.shape[0]
s = np.ones(size)
best_partition = []
for num in range(1, int((size+1.)/2. + 1)):
    for idx_c in combinations(list(range(size)), num):
        s_now = s.copy()
        s_now[list(idx_c)] = -1
        cost = s_now.T.dot(L).dot(s_now)
        if cost < best_cost-tol:
            best_cost = cost
            best_partition = [s_now]
        elif abs(cost-best_cost) < tol:
            best_partition.append(s_now)

print("Best cost:", best_cost)
print("Best partition:", best_partition)

In [ ]:
# s = best_partition[0]

idx_c = 0
for s in best_partition:

    nodec = [1 if s[i] > 0 else 0 for i in range(len(s))]
    # Partition
    # draw_sibigraph_nodec(Gb, nodec, cmap_name='coolwarm', figname=lab_G+lab_p+'-exact{}'.format(idx_c), figsize=figsize)
    # Switching
    # update the labels
    node_c = [nodec][0]

    # s = [1 if node_c[i] == 1 else -1 for i in range(len(node_c))]
    S = np.diag(s)
    W_s = S.dot(W.dot(S))
    B_s = W_s[:len(edges1), :][:, -len(edges2):]

    labels_p1s = []
    labels_p2s = []
    for i in range(len(edges1)):
        if node_c[i] == 1:
            labels_p1s.append(labels_p1[i])
        else:
            e = edges1[i]
            labels_p1s.append('({},{})'.format(e[1], e[0]))
    for j in range(len(edges2)):
        if node_c[j+len(edges1)] == 1:
            labels_p2s.append(labels_p2[j])
        else:
            e = edges2[j]
            labels_p2s.append('({},{})'.format(e[1]+G1.number_of_nodes(), e[0]+G1.number_of_nodes()))

    Gb_s = build_sibigraph(B_s, labels_p1s, labels_p2s)
    draw_sibigraph(Gb_s, figname=lab_G+'-switch'+lab_p+'-exact{}'.format(idx_c), figsize=figsize_b)
    draw_sibigraph_node2c(Gb_s, node_c, figname=lab_G+'-switch'+lab_p+'-exact{}'.format(idx_c), figsize=figsize_b)

    idx_c += 1

The two solutions we found are exact (there are equivalent solutions in the above).

### Linking centrality

In [ ]:
# compute the linking centrality
cmap = 'plasma'
matrix = Lam_gli.copy()
# For G1: absolute row sum (assume one value per edge, e.g., left-aligned)
row_sums = np.abs(matrix).sum(axis=1)
# For G2: absolute column sum (one per edge)
col_sums = np.abs(matrix).sum(axis=0)

pos1 = nx.get_node_attributes(G1, 'pos')
# Graph 2
pos2 = nx.get_node_attributes(G2, 'pos')

plot_networkx_colore(G1, G2, pos1, pos2, row_sums, col_sums, camera=camera,
                     figsize=figsize_g, lab_name=lab_G+'-cent', cmap=cmap,
                     xval=.82, cbar_len=0.5, cbar_thk=15)